# Bond Pricing and Yield Curves
In this notebook, I will be using Python to explore bond pricing and yield curve construction. I will begin by building a pricing engine to price bonds, and then use this to solve the yield to maturity for a given price. From there, I will calculate some key risk measurements for fixed income like duration and convexity. I will then use this to approximate a price change for a given yield shift.

On the yield curve side, I will take Australian Government Bond (AGB) data from the RBA and use it to build a yield curve using interpolation. From this, I will also calculate a forward rates curve, and also use it to price bonds of the curve. 

In [15]:
# Importing packages

import numpy as np
import pandas as pd
import matplotlib as plt
from scipy.optimize import brentq
from datetime import datetime
from dateutil.relativedelta import relativedelta
from IPython.display import Image

# Bond Pricing
First I will begin by creating a bond pricing engine. For some given cash flows/coupon, yield, face value and term to maturity, the price of a bond will be calculated. The following forumla will be used to calculate the price of the bond, which sums the present value of the two bond cashflows, being the face value and coupons:

$$ P = \sum_{t = 1}^{N} \frac{C_t}{(1+y)^t} + \frac{F}{(1+y)^N}. $$

Where:
* $F$ is the face value of the bond, or the amount the bond pays until maturity
* $C_t$ is the coupon payment at time period t, where $C_t = F * i$ and $i$ is the annual coupon rate
* $n$ is the total number of years to maturity
* $y$ is the required discount rate or yield to maturity

The above formula is for annual compounding periods. For a lot of bonds, coupon periods are more frequent than annually, for example, AGBs compounded semi-annually. The formula for bonds that are compounded more frequently can be adjusted to the following, where $m$ is the frequency of coupon payments per year.:

$$  P = \sum_{t = 1}^{m \cdot N} \frac{C/m}{(1+y/m)^t} + \frac{F}{(1+y/m)^{m \cdot N}}. $$

For a zero-coupon bond, the formula price automatically becomes the discounted face value of the bond (no need for an additional formula in our code). 

In [2]:
def bond_price(face_value, coupon_rate, ytm, years, frequency):
  # Adjust the compounding periods to at least 1 to prevent division by zero
  if frequency <= 0:
    frequency = 1
      
  # Calculate the coupon and amount of payments
  coupon = face_value * coupon_rate / frequency
  periods = frequency * years
  
  # Calculate the discount rate, adjusted for frequency of coupon payments
  r = ytm / frequency
  
  # Calculate the present value of the face value and coupons
  pv_face = face_value / (1 + ytm / frequency)**(periods)
  t = np.arange(1, periods + 1) # amount of coupon payments
  pv_coupons = np.sum(coupon / (1 + r)**t)
  
  # Return price
  return pv_face + pv_coupons

For example, a bond with a face value of \$100, an annual coupon rate of 5% with a market discount rate/yield to maturity of 3% for a 10 year bond and semi-annual compounding, we calculate the bond should be priced at \$117.17 (to two decimal places)

In [3]:
# Outlining the features of the bond
face_value = 100
coupon_rate = 0.05
ytm = 0.03
years = 10
frequency = 2

# Calculating the price
price = bond_price(face_value, coupon_rate, ytm, years, frequency)
print(f"Price: ${price:.2f}")

Price: $117.17


## Yield to maturity
We can calculate the yield to maturity of a bond by observing the bond's market price, then backing out the yield to maturity, which we will do in this section of the notebook. 

It is important to note that when calculating the theoretical price of a bond, an appropriate discount rate is selected, called the required rate of return. This is calculated as the sum of the following components:

$$ \text{Discount Rate} = \text{Risk-Free Rate} + \text{Default Risk Premium} + \text{Liquidity Premium} + \text{Term Premium}. $$

* The risk-free rate is the yield of a government bond with the exact same maturity.
* Credit risk is the extra yield to compensate investors for the default risk of a company, This can be caulculated by checking the bond issuer's credit rating, then using relevant credit spreads.
* Liquidity premium is the extra yield if the bond is difficult to sell quickly/convert to cash without impacting the market price. 
* Term premium (also known as interest rate risk premium) is the additional compensation required for investing money in a bond, exposing inverstors to future interest rater fluctuations. This can already be considered in the risk-free yield curve.

Now with all of that said, we can begin to find the yield of maturity (YTM) given a price observed in the market. To do this, we will use the scipy solver function as it is hard to invert the above price equation to find the YTM. Specifically, we will find use Brent's method to find the root of the equation. Knowing all other variables exluding the YTM, we can find the YTM that makes the bond's price equal its given price, in other words finding the root.

In [4]:
def ytm_solver(price, face_value, coupon_rate, years, frequency):
  objective = lambda y: bond_price(face_value, coupon_rate, y, years, frequency) - price
  return brentq(objective, 1e-6, 2.0)
  

For the above hypothetical bond, we calculate that the YTM is 2.9999%. Compared to the YTM of 3% used in the price calculation, the small discretion is caused by us rounding the price of the bond. 

In [5]:
face_value = 100
coupon_rate = 0.0425
years = 10
frequency = 2
price = 95.678

ytm = ytm_solver(price, face_value, coupon_rate, years, frequency)
print(f"Yield {ytm:.4%}%")

Yield 4.7992%%


# Accrued Interest
Whilst researching and learning about bond pricing formula, I also learn about accrued interest. 

The above formula used for bond pricing assumes that we are standing exactly on a coupon date, with a whole number of periods left. However, bonds can be traded between coupon paymetns, so if we are valuing a coupon partway through a coupon period, we need to discount the first cash flow by a fraction of a period, not a whole period. When pricing a bond, we should include this accrued interest in the price, as for example when a bond is traded between coupon payment dates, the seller of the bond has held the bond for a part of the given coupon period, and thus is entitled to that portion of interest earned. 

Thus, it makes sense to add the accrued interest to the price of a bond calculated on a coupon date to compensate for the interest accrued in the time between coupon dates. If we define the dirty price to be the price paid on a bond (for example in a trade), and the clean price to be the price calculated on a coupon date (i.e. not including accrued interest), we get the following formula to price bonds including accrued interest:

$$ P_{Dirty} = P_{Clean} + \text{Accrued Interest}. $$

Where,

$$ \text{Accrued Interest} = C \cdot (1 - w) $$

and $C$ is the coupon and $w$ is the fraction of the current coupon period left until the next coupon. 

In [ ]:
def bond_price_accrued(face_value, coupon_rate, ytm, years, frequency, last_coupon, next_coupon, settlement_date):
  # Get the clean bond price
  p_clean = bond_price(face_value, coupon_rate, ytm, years, frequency)
  
  # Calculate the accrued interest
  coupon = (face_value * coupon_rate) / frequency
  period_len = (next_coupon - last_coupon).days
  period_frac = (settlement_date - last_coupon).days / period_len # fraction of the period remianing
  
  accrued_interest = ((face_value * coupon_rate) / frequency) *  (period_frac)
  
  return p_clean + accrued_interest

def ytm_solver_accrued(price, face_value, coupon_rate, years, frequency, last_coupon, next_coupon, settlement_date):
  objective = lambda y: bond_price_accrued(face_value, coupon_rate, y, years, frequency, last_coupon, next_coupon, settlement_date) - price

  return brentq(objective, 1e-6, 2.0)

We can apply the above function to an example bond. For this, we use an AGB maturing in October 2036 with code GSBS36. This bond has a coupon rate of 4.25%, with the next coupon date on 21/10/2026, and the last coupon date of 21/04/2026 (it is important to note that ex-dates are more important in this regard, as if the bond is bought after the ex-date the coupon will not be paid to that buyer). As of 07/07/2026, after market close, the last traded price of this bond is \$95.678 with a yield of 4.756%. This price would also include future expectations, and so on, but the valuation price given by the ASX is \$96.737. 

With this information, we can now calculate the price and YTM of the bond using our pricer (including accrued interest) and compare to the market data.

In [ ]:
# Inputs to pricer
face_value = 100
coupon_rate = 0.0425
ytm = 0.04756
years = 10
frequency = 2
last_coupon = pd.Timestamp('2026-04-21')
next_coupon = pd.Timestamp('2026-10-21')
settlement_date = pd.Timestamp('2026-07-07')

# Prices from https://www.asx.com.au/markets/trade-our-cash-market/equity-market-prices/bonds
market_price = 95.678
valuation_price = 96.737

# Find the price of the bonds
price_calc = bond_price_accrued(face_value, coupon_rate, ytm, years, frequency, last_coupon, next_coupon, settlement_date)
price_clean = bond_price(face_value, coupon_rate, ytm, years, frequency)

print(f"Price: ${price_calc:.3f}\nMarket price: ${market_price:.3f}\nValuation price: ${valuation_price:.3f}\nClean Price: ${price_clean:.3f}")

# Solve for the market YTM
ytm_price_calc = ytm_solver_accrued(price_calc, face_value, coupon_rate, years, frequency, last_coupon, next_coupon, settlement_date)
ytm_market = ytm_solver_accrued(market_price, face_value, coupon_rate, years, frequency, last_coupon, next_coupon, settlement_date)
ytm_valuation = ytm_solver_accrued(valuation_price, face_value, coupon_rate, years, frequency, last_coupon, next_coupon, settlement_date)

print(f"\nYTM: {ytm_price_calc:.4%}\nMarket YTM: {ytm_market:.4%}\nValuation YTM: {ytm_valuation:.4%}\nASX YTM: {ytm:.4%}")
print(f"ASX YTM used in calculation returns same YTM (so solver is accurate)")


Price: $96.904
Market price: $95.678
Valuation price: $96.737
Clean Price: $96.010

YTM: 4.7560%
Market YTM: 4.9166%
Valuation YTM: 4.7778%
ASX YTM: 4.7560%


We can see that the valuations and YTM are slightly off using this approach. This is becuase by using bond_price(), the pricer discounts every future cash flow as if the trade date is the last coupon date, and then simply adds accrued interest on top. However, if the actual trade date is into the the period, the correct approach is to shift the entire discounted cash-flow stream forward to account for this time discrepancy. All in all, the simple accrued interest calculation from above only approximates the coupon itself accruing, but doesn't correctly discoun the other cash flows, causing some of this error. 

The AOFM has a pricing formula for AGBs, which we will use from now on. 

<div style="text-align: center;">

![alt text](figures/formula_bonds.png "AOFM bond formula")

![alt text](figures/variables_bonds.png "AOFM variables for formula")

</div>

If we break the bracket apart, the formula is valuing the bond as of the next coupon date, then discounting that whole value back to today. 
* The lone $g$ is the next coupon payment, valued at the moment it is paid
* $ g \cdot a_{n}$ is the present value, as of that next coupon date, of all the coupons still to come after it, where $a_n$ is the standard PV-of-annuity formula
* $100v^n$ is the present value, as of the next coupon date, of getting the \$100 face value abck at maturity

So the whole bracket is what the bond is worth if we are standing exactly on the next coupon date. Then, we multiple by $v^{\frac{f}{d}}$ to discount that value back from the next coupon date to today's actual trade date, compounding this discounting factor down over the fraction $f/d$ that remains. Here, $f$ is how many days are left until the next coupon, and $d$ is the full length of the current coupon period. 

We now use this formula to create a new pricing engine.

In [53]:
def bond_price_dirty(face_value, coupon_rate, ytm, frequency, trade_date, last_coupon, next_coupon, maturity_date):
    i = ytm / frequency # discount rate divided by frequency
    v = 1 / (1 + i) # discounting factor
    g = face_value * coupon_rate / frequency # coupon payment
    f_days = (next_coupon - trade_date).days # days until next coupon
    d_days = (next_coupon - last_coupon).days # number of days total in coupon period

    dates = []
    d = last_coupon
    while d < maturity_date: # if there are still remaining coupon periods
        d += relativedelta(months=6) # add six months from most recent coupon date
        dates.append(d) # append this next coupon date
    n = len(dates) - 1  # coupons remaining after the next one

    a_n = (1 - v**n) / i # annuity formula, from AOFM
    return v**(f_days / d_days) * (g * (1 + a_n) + face_value * v **n)

def ytm_solver_dirty(price, face_value, coupon_rate, frequency, trade_date, last_coupon, next_coupon, maturity_date):
    objective = lambda y: bond_price_dirty(face_value, coupon_rate, y, frequency, trade_date, last_coupon, next_coupon, maturity_date) - price
    return brentq(objective, 1e-6, 2.0)
  


And if we apply this pricer to our AGB example, we get some more accurate results to ther market price, and the ASX valuation price. Also, comparing the YTM, the YTM we find is slightly more accurate too.

In [57]:
# Inputs
face_value = 100
coupon_rate = 0.0425
ytm = 0.04756
frequency = 2
last_coupon = pd.Timestamp('2026-04-21')
next_coupon = pd.Timestamp('2026-10-21')
trade_date = pd.Timestamp('2026-07-07')
maturity_date = pd.Timestamp('2036-10-21')
# Prices from https://www.asx.com.au/markets/trade-our-cash-market/equity-market-prices/bonds
market_price = 95.678
valuation_price = 96.737


# Price of the bonds
price_calc = bond_price_dirty(face_value, coupon_rate, ytm, frequency, trade_date, last_coupon, next_coupon, maturity_date)
print(f"Price: ${price_calc:.3f}\nMarket price: ${market_price:.3f}\nValuation price: ${valuation_price:.3f}\n")

# Calculate YTM
ytm_price_calc = ytm_solver_dirty(price_calc, face_value, coupon_rate, frequency, trade_date, last_coupon, next_coupon, maturity_date)
ytm_market     = ytm_solver_dirty(market_price, face_value, coupon_rate, frequency, trade_date, last_coupon, next_coupon, maturity_date)
ytm_valuation  = ytm_solver_dirty(valuation_price, face_value, coupon_rate, frequency, trade_date, last_coupon, next_coupon, maturity_date)

print(f"\nYTM: {ytm_price_calc:.4%}\nMarket YTM: {ytm_market:.4%}\nValuation YTM: {ytm_valuation:.4%}\nASX YTM: {ytm:.4%}")


Price: $96.808
Market price: $95.678
Valuation price: $96.737


YTM: 4.7560%
Market YTM: 4.9007%
Valuation YTM: 4.7651%
ASX YTM: 4.7560%


This pricing formual can be used for any fixed-coupon bond, like UK Gilts, Treasuries, Eurobonds and most vanilla corporate bonds, however some variables will vary by market, so we will need to swap them out. For example, the day count convention may be 30/360, meaning 30 days per month and 360 days a year, instead of counting real calendar days. Thus, a semi-annual coupon period is always treated as exactly 180 days, no matter what. This is standard among most US corporate bonds, and some other bonds too. 

Some of the discrepencies in calculations would be due to market dynamics such as liquidity, bid/ask spreads, future expectations, different pricing formulas (and inputs) used by institutional investors and so on. This is whilst our price does not account for these factors, and uses a more standard, purely mathematical formula. For example, we a pricer using zero-coupon spot rates would be more accurate to what is actually used in the market. 